# Task 1-c — 점수 산출 방식 개선 및 판정기 재비교

## 직전 실험에서 확인된 것

- 근거 문서를 창으로 나눠 전부 훑자 같은 모델의 AUROC가 0.695 → 0.744로 개선되었다.
  절단이 성능 저하의 원인 중 하나였음이 확인되었으나, 목표 기준 0.80에는 여전히 미달한다.
- 환각 유형별 성능에서 **Evident Conflict가 0.641로 가장 낮았다.** 명백한 충돌임에도
  가장 못 잡는 것은 점수 산출 방식의 결함을 시사한다.

## 이 실험의 가설

현재 점수는 `1 − 함의확률`이며 창별 최댓값 함의확률을 사용한다.
충돌형 환각은 근거와 같은 대상을 다루되 값만 다르므로 특정 창에서 함의확률이 높게 나올 수 있고,
최댓값 집계가 이를 정상으로 판정해버린다. **모순(contradiction) 확률을 별도로 반영하면
충돌형 탐지가 개선될 것이다.**

## 비교할 점수 산출 방식

| 방식 | 정의 |
|---|---|
| `1-entail` | 기존 방식. 창별 최대 함의확률의 여집합 |
| `contra` | 창별 최대 모순확률 |
| `contra-entail` | 최대 모순확률 − 최대 함의확률 |
| `mean-entail` | 최댓값 대신 평균 함의확률 사용 (집계 방식 비교) |
| `combo` | (1−최대함의) 와 최대모순의 가중합 |

## 사전 정의 기준

- 판별력: AUROC 0.80 이상 / 지연: 문장당 100ms 이하
- 두 기준을 함께 만족하는 기성 판정기가 없으면 RAGTruth 파인튜닝으로 전환한다.

## 1. 환경 준비 (nltk 데이터 포함 — MiniCheck 실패 원인이었음)

In [ ]:
!pip -q install pysbd sentencepiece scikit-learn pandas matplotlib nltk
!git clone --depth 1 https://github.com/ParticleMedia/RAGTruth.git 2>/dev/null || echo 'already cloned'

import nltk
for pkg in ['punkt', 'punkt_tab']:
    try: nltk.download(pkg, quiet=True)
    except Exception as e: print(pkg, 'download 실패:', e)

import json, time, random, gc
import numpy as np, pandas as pd, torch, transformers, pysbd
from sklearn.metrics import average_precision_score, roc_auc_score

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('transformers', transformers.__version__, '| device', DEVICE)

## 2. 데이터 준비 (직전 실험과 동일 시드·동일 표본)

In [ ]:
BASE = 'RAGTruth/dataset'
responses = [json.loads(l) for l in open(f'{BASE}/response.jsonl')]
sources   = {json.loads(l)['source_id']: json.loads(l) for l in open(f'{BASE}/source_info.jsonl')}
for r in responses:
    r['task_type'] = sources[r['source_id']]['task_type']

def build_reference(src):
    si, tt = src['source_info'], src['task_type']
    if tt == 'QA':
        return f"Question: {si.get('question','')}\n{si.get('passages','')}"
    if tt == 'Summary':
        return si if isinstance(si, str) else json.dumps(si, ensure_ascii=False)
    parts = []
    for k, v in (si.items() if isinstance(si, dict) else []):
        parts.append(f"{k}: {json.dumps(v, ensure_ascii=False) if isinstance(v,(dict,list)) else v}")
    return '\n'.join(parts)

N_PER_TASK = 120
dev_pool = [r for r in responses if r['split'] == 'train']
dev = []
for tt in ['QA', 'Summary', 'Data2txt']:
    cand = [r for r in dev_pool if r['task_type'] == tt]
    dev += random.sample(cand, min(N_PER_TASK, len(cand)))

seg = pysbd.Segmenter(language='en', clean=False, char_span=True)

def split_sentences(t):
    return [(s.start, s.end, t[s.start:s.end]) for s in seg.segment(t) if s.sent.strip()]

def label_sentence(a, b, labels):
    return int(any(min(b, l['end']) > max(a, l['start']) for l in labels))

def dominant_label_type(a, b, labels):
    best, ov_best = None, 0
    for l in labels:
        ov = max(0, min(b, l['end']) - max(a, l['start']))
        if ov > ov_best: best, ov_best = l.get('label_type'), ov
    return best

recs = []
for r in dev:
    labels = r.get('labels') or []
    for i, (a, b, s) in enumerate(split_sentences(r['response'])):
        recs.append({'resp_id': r['id'], 'source_id': r['source_id'], 'task_type': r['task_type'],
                     'sent_idx': i, 'sentence': s,
                     'y': label_sentence(a, b, labels),
                     'label_type': dominant_label_type(a, b, labels)})
df = pd.DataFrame(recs)
df['reference'] = df.source_id.map(lambda sid: build_reference(sources[sid]))
RANDOM_AUPRC = df.y.mean()
print(f'문장 {len(df):,}개 | 양성 {df.y.sum():,}개 ({RANDOM_AUPRC*100:.1f}%)')

## 3. 창 분할 NLI — 세 확률을 모두 보관

추론은 한 번만 수행하고 entailment / neutral / contradiction 확률을 모두 저장한 뒤,
점수 산출 방식만 바꿔가며 비교한다. 방식마다 재추론하지 않으므로 시간이 절약된다.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

CKPT = 'microsoft/deberta-large-mnli'
tok = AutoTokenizer.from_pretrained(CKPT)
model = AutoModelForSequenceClassification.from_pretrained(CKPT).to(DEVICE).eval()

id2label = {i: l.lower() for i, l in model.config.id2label.items()}
IDX = {k: [i for i, l in id2label.items() if l.startswith(p)][0]
       for k, p in [('ent','entail'), ('neu','neutral'), ('con','contradic')]}
print('라벨 인덱스:', {k: id2label[v] for k, v in IDX.items()})

WIN_TOKENS, STRIDE = 400, 200

def make_windows(text):
    ids = tok(text, add_special_tokens=False)['input_ids']
    if len(ids) <= WIN_TOKENS:
        return [text]
    out = []
    for i in range(0, len(ids), STRIDE):
        chunk = ids[i:i+WIN_TOKENS]
        if not chunk: break
        out.append(tok.decode(chunk))
        if i + WIN_TOKENS >= len(ids): break
    return out

# 근거 문서별 창을 미리 만들어 재사용 (같은 source_id가 여러 문장에서 반복됨)
win_cache = {sid: make_windows(build_reference(sources[sid])) for sid in df.source_id.unique()}
print('평균 창 개수: %.2f' % np.mean([len(v) for v in win_cache.values()]))

In [ ]:
BATCH = 32

@torch.no_grad()
def probs_batch(prem, hyp):
    enc = tok(prem, hyp, truncation=True, max_length=512,
              padding=True, return_tensors='pt').to(DEVICE)
    return model(**enc).logits.softmax(-1).cpu().numpy()

# 모든 (문장 × 창) 쌍을 펼쳐 한 번에 추론
flat_p, flat_h, owner = [], [], []
for idx, row in enumerate(df.itertuples()):
    for w in win_cache[row.source_id]:
        flat_p.append(w); flat_h.append(row.sentence); owner.append(idx)
print(f'총 추론 쌍: {len(flat_p):,}')

if DEVICE == 'cuda': torch.cuda.synchronize()
t0 = time.perf_counter()
P = []
for i in range(0, len(flat_p), BATCH):
    P.append(probs_batch(flat_p[i:i+BATCH], flat_h[i:i+BATCH]))
if DEVICE == 'cuda': torch.cuda.synchronize()
elapsed = time.perf_counter() - t0
P = np.vstack(P)
owner = np.array(owner)
MS_PER_SENT = elapsed * 1000 / len(df)
print(f'추론 완료: {elapsed:.1f}초 | 문장당 {MS_PER_SENT:.1f} ms')

In [ ]:
# 창별 확률을 문장 단위로 집계
n = len(df)
ent_max = np.zeros(n); ent_sum = np.zeros(n); cnt = np.zeros(n)
con_max = np.zeros(n); con_sum = np.zeros(n)

for o, p in zip(owner, P):
    e, c = p[IDX['ent']], p[IDX['con']]
    ent_max[o] = max(ent_max[o], e); ent_sum[o] += e
    con_max[o] = max(con_max[o], c); con_sum[o] += c
    cnt[o] += 1
ent_mean, con_mean = ent_sum / cnt, con_sum / cnt

SCORING = {
    '1-entail (기존)':  1.0 - ent_max,
    'contra':           con_max,
    'contra-entail':    con_max - ent_max,
    'mean-entail':      1.0 - ent_mean,
    'combo(0.5/0.5)':   0.5*(1.0-ent_max) + 0.5*con_max,
    'combo(0.7/0.3)':   0.7*(1.0-ent_max) + 0.3*con_max,
}

y = df.y.values
rows = []
for name, s in SCORING.items():
    rows.append({'scoring': name,
                 'AUROC': roc_auc_score(y, s),
                 'AUPRC': average_precision_score(y, s),
                 'lift': average_precision_score(y, s)/RANDOM_AUPRC})
score_res = pd.DataFrame(rows).sort_values('AUROC', ascending=False).reset_index(drop=True)
score_res.round(3)

## 4. 가설 검증 — 충돌형 환각이 실제로 개선되었는가

In [ ]:
neg_mask = (df.y == 0).values
print(f"{'scoring':18}", end='')
types = [t for t in df.label_type.dropna().unique()
         if ((df.y==1) & (df.label_type==t)).sum() >= 10]
for t in types: print(f'{t[:22]:>24}', end='')
print()

for name, s in SCORING.items():
    print(f'{name:18}', end='')
    for t in types:
        pos_mask = ((df.y==1) & (df.label_type==t)).values
        m = pos_mask | neg_mask
        print(f'{roc_auc_score(y[m], s[m]):>24.3f}', end='')
    print()

print('\n→ Evident Conflict 열에서 1-entail 대비 contra 계열이 개선되면 가설이 지지된다.')

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8,4))
x = np.arange(len(types)); w = 0.8/len(SCORING)
for k, (name, s) in enumerate(SCORING.items()):
    vals = []
    for t in types:
        m = (((df.y==1) & (df.label_type==t)).values) | neg_mask
        vals.append(roc_auc_score(y[m], s[m]))
    ax.bar(x + k*w, vals, w, label=name)
ax.set_xticks(x + 0.4); ax.set_xticklabels([t[:20] for t in types], fontsize=8)
ax.axhline(0.5, color='gray', ls='--', lw=1)
ax.set_ylabel('AUROC'); ax.set_title('Scoring method x hallucination type')
ax.legend(fontsize=7); plt.tight_layout(); plt.show()

## 5. 근거 훑는 범위와 성능의 관계

창을 앞에서부터 k개만 사용했을 때의 성능을 계산한다. 추가 추론 없이 기존 결과를 재집계하므로 비용이 없다.
창 수를 늘릴수록 성능이 오르되 지연도 비례해 증가한다면, **근거를 얼마나 훑을지**가
적응형 조절의 대상이 될 수 있다.

In [ ]:
BEST = score_res.iloc[0].scoring
use_contra = 'contra' in BEST or 'combo' in BEST
print('기준 점수 방식:', BEST)

# 문장별 창 순서 인덱스 부여
order = np.zeros(len(owner), dtype=int)
seen = {}
for i, o in enumerate(owner):
    seen[o] = seen.get(o, -1) + 1
    order[i] = seen[o]

max_win = int(order.max()) + 1
rows = []
for k in range(1, max_win + 1):
    sel = order < k
    e = np.zeros(n); c = np.zeros(n)
    for o, p in zip(owner[sel], P[sel]):
        e[o] = max(e[o], p[IDX['ent']]); c[o] = max(c[o], p[IDX['con']])
    s = (0.5*(1-e) + 0.5*c) if use_contra else (1 - e)
    rows.append({'windows_used': k,
                 'AUROC': roc_auc_score(y, s),
                 'AUPRC': average_precision_score(y, s),
                 'rel_cost': sel.sum()/len(owner)})
win_res = pd.DataFrame(rows)
print(win_res.round(3).to_string(index=False))

fig, ax = plt.subplots(figsize=(6,4))
ax.plot(win_res.rel_cost*MS_PER_SENT, win_res.AUROC, 'o-')
for _, r in win_res.iterrows():
    ax.annotate(f"{int(r.windows_used)}창", (r.rel_cost*MS_PER_SENT, r.AUROC),
                textcoords='offset points', xytext=(5,4), fontsize=8)
ax.set_xlabel('estimated latency per sentence (ms)'); ax.set_ylabel('AUROC')
ax.set_title('How much of the reference to scan'); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

In [ ]:
del model; gc.collect()
if DEVICE == 'cuda': torch.cuda.empty_cache()
print('DeBERTa 해제 완료')

## 6. 전용 판정기 재시도 (MiniCheck / HHEM)

MiniCheck는 직전 실행에서 nltk 리소스 누락으로 실패했다. 상단에서 punkt_tab을 설치했으므로 재시도한다.
HHEM은 로드 단계 실패 여부와 원인을 확인한다.

In [ ]:
!pip -q install "minicheck @ git+https://github.com/Liyan06/MiniCheck.git@main" 2>&1 | tail -2

In [ ]:
extra = []

# --- MiniCheck ---
try:
    from minicheck.minicheck import MiniCheck
    mc = MiniCheck(model_name='flan-t5-large', enable_prefix_caching=False)
    docs, claims = df.reference.tolist(), df.sentence.tolist()
    t0 = time.perf_counter()
    _, raw_prob, _, _ = mc.score(docs=docs, claims=claims)
    el = time.perf_counter() - t0
    s = 1.0 - np.asarray(raw_prob)
    extra.append({'judge': 'MiniCheck-FlanT5-L',
                  'AUROC': roc_auc_score(y, s), 'AUPRC': average_precision_score(y, s),
                  'ms_per_sentence': el*1000/len(df)})
    globals()['minicheck_scores'] = s
    print(f"MiniCheck  AUROC {extra[-1]['AUROC']:.3f}  AUPRC {extra[-1]['AUPRC']:.3f}  "
          f"{extra[-1]['ms_per_sentence']:.1f} ms/문장")
    del mc
except Exception as e:
    print(f'[MiniCheck 실패] {type(e).__name__}: {str(e)[:300]}')
finally:
    gc.collect()
    if DEVICE == 'cuda': torch.cuda.empty_cache()

In [ ]:
# --- HHEM 2.1 open ---
try:
    hhem = AutoModelForSequenceClassification.from_pretrained(
        'vectara/hallucination_evaluation_model', trust_remote_code=True).to(DEVICE).eval()
    pairs = list(zip(df.reference.tolist(), df.sentence.tolist()))
    out = []
    t0 = time.perf_counter()
    with torch.no_grad():
        for i in range(0, len(pairs), 16):
            sc = hhem.predict(pairs[i:i+16])
            sc = sc.detach().cpu().numpy() if torch.is_tensor(sc) else np.asarray(sc)
            out.extend(np.ravel(sc).tolist())
    el = time.perf_counter() - t0
    s = 1.0 - np.array(out)
    extra.append({'judge': 'HHEM-2.1-open',
                  'AUROC': roc_auc_score(y, s), 'AUPRC': average_precision_score(y, s),
                  'ms_per_sentence': el*1000/len(df)})
    globals()['hhem_scores'] = s
    print(f"HHEM  AUROC {extra[-1]['AUROC']:.3f}  AUPRC {extra[-1]['AUPRC']:.3f}  "
          f"{extra[-1]['ms_per_sentence']:.1f} ms/문장")
    del hhem
except Exception as e:
    print(f'[HHEM 실패] {type(e).__name__}: {str(e)[:300]}')
finally:
    gc.collect()
    if DEVICE == 'cuda': torch.cuda.empty_cache()

## 7. 종합 및 사전 기준 판정

In [ ]:
AUROC_MIN, MS_MAX = 0.80, 100.0

final = pd.DataFrame(
    [{'judge': f'DeBERTa-MNLI 창분할 [{r.scoring}]', 'AUROC': r.AUROC, 'AUPRC': r.AUPRC,
      'ms_per_sentence': MS_PER_SENT} for r in score_res.itertuples()] + extra
).sort_values('AUROC', ascending=False).reset_index(drop=True)

print(f'무작위 기준선: AUPRC {RANDOM_AUPRC:.3f}, AUROC 0.500')
print(f'통과 기준: AUROC ≥ {AUROC_MIN}, 지연 ≤ {MS_MAX}ms\n')
for _, r in final.iterrows():
    a = '✓' if r.AUROC >= AUROC_MIN else '✗'
    l = '✓' if r.ms_per_sentence <= MS_MAX else '✗'
    print(f'{r.judge:42} AUROC {r.AUROC:.3f}{a}  {r.ms_per_sentence:7.1f}ms{l}  '
          f'lift {r.AUPRC/RANDOM_AUPRC:.2f}x')

passed = final[(final.AUROC >= AUROC_MIN) & (final.ms_per_sentence <= MS_MAX)]
print()
if len(passed):
    print(f'결론: {passed.iloc[0].judge} 채택 → 이 판정기로 검증 단위 실험을 재수행')
elif (final.AUROC >= AUROC_MIN).any():
    b = final[final.AUROC >= AUROC_MIN].iloc[0]
    print(f'결론: 정확도 기준은 {b.judge}가 만족하나 지연 기준 초과 '
          f'({b.ms_per_sentence:.0f}ms) → 경량화가 과제로 확정됨')
else:
    print('결론: 정확도 기준을 만족하는 기성 판정기 없음 → RAGTruth 파인튜닝으로 전환')

final.to_csv('judge_comparison_v2.csv', index=False)
score_res.to_csv('scoring_ablation.csv', index=False)
win_res.to_csv('window_budget.csv', index=False)
print('\n저장: judge_comparison_v2.csv, scoring_ablation.csv, window_budget.csv')